####Imports & Setups

In [59]:
import pandas as pd
import numpy as np
import json
import re
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from tqdm.auto import tqdm
import time

# NLP libraries
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
import spacy

# Translation
from deep_translator import GoogleTranslator
from langdetect import detect, LangDetectException

In [60]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

####Load Data & Configs

In [61]:
main_directory = '/content/drive/MyDrive/Barnabus Project/'
df_raw = pd.read_csv(f'{main_directory}data/raw/original_data.csv')

# Load column mapping
with open(f'{main_directory}data/processed/column_mapping.json', 'r') as f:
    column_mapping = json.load(f)

text_columns = column_mapping['text_columns']
rating_columns = column_mapping['rating_columns']

print(f"Text columns: {text_columns}")
print(f"Rating columns: {rating_columns}")

Text columns: ['comment']
Rating columns: ['rating']


In [62]:
df_raw = df_raw.dropna().reset_index(drop=True)
df_raw.head()

,index,rating,comment
0,0,2.0,Ich bin franzose und bin seit ein paar Wochen ...
1,1,6.0,Dieser Arzt ist das unmöglichste was mir in me...
2,2,1.0,Hatte akute Beschwerden am Rücken. Herr Magura...
3,3,1.0,Nachdem ich in der Klinik nur ungenaue Angaben...
4,4,1.0,"Frau Dr. Vetter kenne ich seit vielen Jahren, ..."


####Sampling

In [63]:
SAMPLE_SIZE = 6000

if len(df_raw) >= SAMPLE_SIZE:
    df_sample = df_raw.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"Sampled {SAMPLE_SIZE} rows from {len(df_raw)} total rows")
else:
    df_sample = df_raw.copy()
    print(f"Dataset has fewer than {SAMPLE_SIZE} rows, using all {len(df_raw)} rows")

df_sample.to_csv(f'{main_directory}data/processed/sampled_data.csv', index=False)

Sampled 6000 rows from 429736 total rows


####Data Cleanings & Utilities

In [64]:
def clean_text(text):
    """
    Clean text data:
    - Convert to string
    - Remove URLs
    - Remove emails
    - Remove extra whitespace
    - Remove special characters (keep basic punctuation)
    """
    if pd.isna(text):
        return ""

    text = str(text)

    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    text = re.sub(r'\S+@\S+', '', text)

    text = re.sub(r'\s+', ' ', text)

    text = re.sub(r'[^\w\s.,!?;:\-äöüßÄÖÜéèêëàâôûîïçÉÈÊËÀÂÔÛÎÏÇ]', '', text)

    return text.strip()


def anonymize_text(text):
    """
    Basic anonymization:
    - Remove phone numbers
    - Remove potential patient IDs
    """
    if pd.isna(text) or text == "":
        return text

    text = re.sub(r'\+?\d[\d\s\-\(\)]{7,}\d', '[PHONE]', text)

    text = re.sub(r'\b[A-Z0-9]{5,}\b', '[ID]', text)

    return text


def detect_language(text):
    """Detect language of text"""
    try:
        if pd.isna(text) or text == "" or len(text) < 10:
            return 'unknown'
        return detect(text)
    except LangDetectException:
        return 'unknown'

####Apply Cleaning

In [65]:
if text_columns:
    primary_text_col = text_columns[0]  # Use first text column
else:
    text_lengths = {}
    for col in df_sample.select_dtypes(include=['object']).columns:
        avg_len = df_sample[col].dropna().astype(str).str.len().mean()
        text_lengths[col] = avg_len
    primary_text_col = max(text_lengths, key=text_lengths.get)

df_work = df_sample.copy()

print("Cleaning text...")
df_work['text_original'] = df_work[primary_text_col].copy()
df_work['text_cleaned'] = df_work[primary_text_col].apply(clean_text)
df_work['text_cleaned'] = df_work['text_cleaned'].apply(anonymize_text)

# Remove empty texts
initial_count = len(df_work)
df_work = df_work[df_work['text_cleaned'].str.len() > 10].reset_index(drop=True)
removed = initial_count - len(df_work)

# Detect original language
df_work['language_original'] = df_work['text_cleaned'].apply(detect_language)

lang_dist = df_work['language_original'].value_counts()
print("Language distribution:")
print(lang_dist)

Cleaning text...
Language distribution:
language_original
de    5992
no       3
en       3
sv       1
af       1
Name: count, dtype: int64


####Rating Extraction & Normalization

In [ ]:
if rating_columns:
    rating_col = rating_columns[0]
    df_work['rating'] = pd.to_numeric(df_work[rating_col], errors='coerce')

    max_rating = df_work['rating'].max()
    if max_rating > 5:
        df_work['rating_normalized'] = (df_work['rating'] / max_rating) * 5
    else:
        df_work['rating_normalized'] = df_work['rating']

    print("Rating statistics:")
    print(df_work['rating_normalized'].describe())
else:
    print("No rating column found, will use sentiment analysis only")
    df_work['rating_normalized'] = None

Rating statistics:
count    6000.000000
mean        1.314583
std         1.135793
min         0.833333
25%         0.833333
50%         0.833333
75%         0.833333
max         5.000000
Name: rating_normalized, dtype: float64


####Translation to Multiple Langs

In [ ]:
TARGET_SIZE = min(6000, len(df_work))
ROWS_PER_LANG = TARGET_SIZE // 4

print(f"Target distribution:")
print(f"   Total rows: {TARGET_SIZE}")
print(f"   Rows per language: {ROWS_PER_LANG}")
print(f"   German (de): {ROWS_PER_LANG}")
print(f"   English (en): {ROWS_PER_LANG}")
print(f"   French (fr): {ROWS_PER_LANG}")
print(f"   Arabic (ar): {ROWS_PER_LANG}")

Target distribution:
   Total rows: 6000
   Rows per language: 1500
   German (de): 1500
   English (en): 1500
   French (fr): 1500
   Arabic (ar): 1500


In [ ]:
if len(df_work) < TARGET_SIZE:
    print(f"Warning: Only {len(df_work)} rows available, adjusting target")
    TARGET_SIZE = len(df_work)
    ROWS_PER_LANG = TARGET_SIZE // 4

In [ ]:
df_work = df_work.head(TARGET_SIZE).reset_index(drop=True)

df_work['target_language'] = None
df_work.loc[0:ROWS_PER_LANG-1, 'target_language'] = 'de'  # German (original)
df_work.loc[ROWS_PER_LANG:ROWS_PER_LANG*2-1, 'target_language'] = 'en'  # English
df_work.loc[ROWS_PER_LANG*2:ROWS_PER_LANG*3-1, 'target_language'] = 'fr'  # French
df_work.loc[ROWS_PER_LANG*3:, 'target_language'] = 'ar'  # Arabic

print(df_work['target_language'].value_counts())

target_language
de    1500
en    1500
fr    1500
ar    1500
Name: count, dtype: int64


####Translation Function

In [ ]:
def translate_text_batch(texts, target_lang, source_lang='de', max_retries=3):
    """
    Translate a batch of texts with retry logic
    """
    results = []

    translator = GoogleTranslator(source=source_lang, target=target_lang)

    for text in tqdm(texts, desc=f"Translating to {target_lang}"):
        if pd.isna(text) or text == "" or len(text) < 5:
            results.append(text)
            continue

        # Try translation with retries
        for attempt in range(max_retries):
            try:
                # GoogleTranslator has 5000 char limit
                if len(text) > 4500:
                    # Split into sentences and translate
                    sentences = sent_tokenize(text)
                    translated_sentences = []
                    for sent in sentences:
                        if len(sent) < 4500:
                            translated_sentences.append(translator.translate(sent))
                        else:
                            # If single sentence too long, truncate
                            translated_sentences.append(translator.translate(sent[:4500]))
                    translated = ' '.join(translated_sentences)
                else:
                    translated = translator.translate(text)

                results.append(translated)
                time.sleep(0.1)  # Rate limiting
                break

            except Exception as e:
                if attempt == max_retries - 1:
                    print(f"\n⚠ Translation failed after {max_retries} attempts: {str(e)[:100]}")
                    results.append(text)  # Keep original on failure
                else:
                    time.sleep(1)  # Wait before retry

    return results

####Perform Translation

In [ ]:
df_work['text_final'] = df_work['text_cleaned'].copy()

# Translate English subset
print("Translating to English...")
en_mask = df_work['target_language'] == 'en'
en_texts = df_work.loc[en_mask, 'text_cleaned'].tolist()
df_work.loc[en_mask, 'text_final'] = translate_text_batch(en_texts, 'en', 'de')

# Translate French subset
print("Translating to French...")
fr_mask = df_work['target_language'] == 'fr'
fr_texts = df_work.loc[fr_mask, 'text_cleaned'].tolist()
df_work.loc[fr_mask, 'text_final'] = translate_text_batch(fr_texts, 'fr', 'de')

# Translate Arabic subset
print("Translating to Arabic...")
ar_mask = df_work['target_language'] == 'ar'
ar_texts = df_work.loc[ar_mask, 'text_cleaned'].tolist()
df_work.loc[ar_mask, 'text_final'] = translate_text_batch(ar_texts, 'ar', 'de')

print("All translations complete!")


# Show samples
print("Sample translations:")
for lang in ['de', 'en', 'fr', 'ar']:
    sample = df_work[df_work['target_language'] == lang].iloc[0]
    print(f"\n{lang.upper()}:")
    print(f"Original (de): {sample['text_cleaned'][:100]}...")
    print(f"Translated: {sample['text_final'][:100]}...")

Translating to English...


Translating to en:   0%|          | 0/1500 [00:00<?, ?it/s]

Translating to French...


Translating to fr:   0%|          | 0/1500 [00:00<?, ?it/s]

Translating to Arabic...


Translating to ar:   0%|          | 0/1500 [00:00<?, ?it/s]


⚠ Translation failed after 3 attempts: Schlechtes Hören-Ohrenreinigung --> No translation was found using the current translator. Try anoth
All translations complete!
Sample translations:

DE:
Original (de): Sehr gut. Und sehr lieb. br  Er hat meinen Zahn gerettet.br  OP ohne Komplikationen....
Translated: Sehr gut. Und sehr lieb. br  Er hat meinen Zahn gerettet.br  OP ohne Komplikationen....

EN:
Original (de): Seit einiger Zeit bin ich Patientin bei Frau Dr. med. Y. Kampmann nachdem meine vorherige Frauenärzt...
Translated: I have been a patient of Dr. for some time now. med. Y. Kampmann after my previous gynecologist reti...

FR:
Original (de): ich bin sehr zufrieden mit Einer tollen be handling wären Meine Rückenschmerzen weg...
Translated: Je suis très satisfait de la bonne maniabilité et mes maux de dos auraient disparu...

AR:
Original (de): Dr. Schubert sowie sein Team sind sehr freundlich und arbeiten fachlich einwandfrei. Praxis ist sehr...
Translated: الدكتور شوبرت وفريقه و

##Stopwords Handling

In [ ]:
stopwords_dict = {
    'de': set(stopwords.words('german')),
    'en': set(stopwords.words('english')),
    'fr': set(stopwords.words('french')),
    'ar': set(stopwords.words('arabic'))
}

for lang, stops in stopwords_dict.items():
    print(f"   {lang}: {len(stops)} stopwords")

# Add custom medical/healthcare stopwords
custom_stopwords = {
    'de': {'arzt', 'ärztin', 'praxis', 'termin', 'patient', 'patientin', 'herr', 'frau', 'dr', 'doktor'},
    'en': {'doctor', 'clinic', 'appointment', 'patient', 'mr', 'mrs', 'dr', 'hospital'},
    'fr': {'docteur', 'médecin', 'clinique', 'rendez-vous', 'patient', 'patiente', 'dr', 'hôpital'},
    'ar': {'طبيب', 'عيادة', 'موعد', 'مريض', 'مستشفى'}
}

for lang in stopwords_dict:
    stopwords_dict[lang].update(custom_stopwords.get(lang, set()))
    print(f"   {lang}: +{len(custom_stopwords.get(lang, set()))} custom stopwords")


def remove_stopwords(text, language):
    """Remove stopwords from text"""
    if pd.isna(text) or text == "":
        return text

    stops = stopwords_dict.get(language, set())

    # Tokenize
    words = word_tokenize(text.lower())

    # Remove stopwords and short words
    filtered = [w for w in words if w not in stops and len(w) > 2]

    return ' '.join(filtered)


df_work['text_no_stopwords'] = df_work.apply(
    lambda row: remove_stopwords(row['text_final'], row['target_language']),
    axis=1
)

# Show impact
print("Stopword removal impact:")
avg_before = df_work['text_final'].str.split().str.len().mean()
avg_after = df_work['text_no_stopwords'].str.split().str.len().mean()
reduction = ((avg_before - avg_after) / avg_before * 100)
print(f"   Average words before: {avg_before:.1f}")
print(f"   Average words after: {avg_after:.1f}")
print(f"   Reduction: {reduction:.1f}%")

   de: 232 stopwords
   en: 198 stopwords
   fr: 157 stopwords
   ar: 701 stopwords
   de: +10 custom stopwords
   en: +8 custom stopwords
   fr: +8 custom stopwords
   ar: +5 custom stopwords
Stopword removal impact:
   Average words before: 56.1
   Average words after: 31.1
   Reduction: 44.6%


##Saved Processed Data

In [ ]:
final_columns = [
    'text_original',
    'text_cleaned',
    'text_final',
    'text_no_stopwords',
    'target_language',
    'language_original',
]

if 'rating_normalized' in df_work.columns:
    final_columns.append('rating_normalized')

df_final = df_work[final_columns].copy()

# Save processed data
df_final.to_csv(f'{main_directory}data/processed/processed_multilingual.csv', index=False)
print(f"Saved processed data: {df_final.shape}")

# Save metadata
metadata = {
    'total_rows': len(df_final),
    'languages': df_final['target_language'].value_counts().to_dict(),
    'random_state': RANDOM_STATE,
    'processing_date': pd.Timestamp.now().isoformat(),
    'stopwords_removed': True,
    'columns': list(df_final.columns),
    'avg_text_length_before_stopwords': avg_before,
    'avg_text_length_after_stopwords': avg_after
}

with open(f'{main_directory}data/processed/processing_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

Saved processed data: (6000, 7)


##Quality Check

In [ ]:
print("PROCESSED DATA SUMMARY:")
print(f"   Total rows: {len(df_final)}")
print(f"   Total columns: {len(df_final.columns)}")
print(f"   Languages: {df_final['target_language'].nunique()}")

print("Language distribution (final):")
lang_counts = df_final['target_language'].value_counts()
for lang, count in lang_counts.items():
    pct = (count / len(df_final) * 100)
    print(f"   {lang}: {count:,} ({pct:.1f}%)")

if 'rating_normalized' in df_final.columns:
    print("Rating statistics:")
    print(df_final['rating_normalized'].describe())

print("Text length statistics (characters):")
text_stats = df_final['text_final'].str.len().describe()
print(text_stats)

print("Text length statistics (words):")
word_stats = df_final['text_no_stopwords'].str.split().str.len().describe()
print(word_stats)

print("\n❓ Missing values check:")
print(df_final.isnull().sum())

print("PHASE 2 COMPLETE: DATA PREPROCESSED & TRANSLATED")
print("\nNext steps:")
print(f"1. Review {main_directory}data/processed/processed_multilingual.csv")
print("2. Proceed to 03_embedding_and_nlp.ipynb")
print("3. We will create embeddings and perform clustering")

PROCESSED DATA SUMMARY:
   Total rows: 6000
   Total columns: 7
   Languages: 4
Language distribution (final):
   de: 1,500 (25.0%)
   en: 1,500 (25.0%)
   fr: 1,500 (25.0%)
   ar: 1,500 (25.0%)
Rating statistics:
count    6000.000000
mean        1.314583
std         1.135793
min         0.833333
25%         0.833333
50%         0.833333
75%         0.833333
max         5.000000
Name: rating_normalized, dtype: float64
Text length statistics (characters):
count    6000.000000
mean      339.337333
std       309.099231
min        11.000000
25%       134.000000
50%       258.000000
75%       438.000000
max      2117.000000
Name: text_final, dtype: float64
Text length statistics (words):
count    6000.000000
mean       31.076500
std        27.961479
min         1.000000
25%        13.000000
50%        24.000000
75%        40.000000
max       217.000000
Name: text_no_stopwords, dtype: float64

❓ Missing values check:
text_original        0
text_cleaned         0
text_final           0
text_n

In [ ]:
df_final.head()

,text_original,text_cleaned,text_final,text_no_stopwords,target_language,language_original,rating_normalized
0,Sehr gut. Und sehr lieb. <br />\nEr hat meinen...,Sehr gut. Und sehr lieb. br Er hat meinen Zah...,Sehr gut. Und sehr lieb. br Er hat meinen Zah...,gut lieb zahn gerettet.br komplikationen,de,de,0.833333
1,Es findet so gut wie keine Kommunikation zwisc...,Es findet so gut wie keine Kommunikation zwisc...,Es findet so gut wie keine Kommunikation zwisc...,findet gut kommunikation statt erfährt werte k...,de,de,4.166667
2,"Sehr freundliches Personal, entspannter, nette...","Sehr freundliches Personal, entspannter, nette...","Sehr freundliches Personal, entspannter, nette...",freundliches personal entspannter netter abgeh...,de,de,0.833333
3,"Ich kam durch Empfehlung einer Freundin, die s...","Ich kam durch Empfehlung einer Freundin, die s...","Ich kam durch Empfehlung einer Freundin, die s...",kam empfehlung freundin dr. rauscher sympathis...,de,de,0.833333
4,Aufgrund meines Umzugs nach München durfte ich...,Aufgrund meines Umzugs nach München durfte ich...,Aufgrund meines Umzugs nach München durfte ich...,aufgrund umzugs münchen durfte dr. salamander ...,de,de,0.833333
